## **Experiment No : 09**

### **Chi-Squared Goodness-Of-Fit Test**

In our study of t-tests, we introduced the one-way t-test to check whether a sample mean differs from the an expected (population) mean. The chi-squared goodness-of-fit test is an analog of the one-way t-test for categorical variables: it tests whether the distribution of sample categorical data matches an expected distribution. For example, you could use a chi-squared goodness-of-fit test to check whether the race demographics of members at your church or school match that of the entire U.S. population or whether the computer browser preferences of your friends match those of Internet uses as a whole.

When working with categorical data, the values themselves aren't of much use for statistical testing because categories like "male", "female," and "other" have no mathematical meaning. Tests dealing with categorical variables are based on variable counts instead of the actual value of the variables themselves.

Let's generate some fake demographic data for U.S. and Minnesota and walk through the chi-square goodness of fit test to check whether they are different:

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [ ]:
national = pd.DataFrame(["white"]*100000 + ["hispanic"]*60000 +\
                        ["black"]*50000 + ["asian"]*15000 + ["other"]*35000)


minnesota = pd.DataFrame(["white"]*600 + ["hispanic"]*300 + \
                         ["black"]*250 +["asian"]*75 + ["other"]*150)

national_table = pd.crosstab(index=national[0], columns="count")
minnesota_table = pd.crosstab(index=minnesota[0], columns="count")

print( "National")
print(national_table)
print(" ")
print( "Minnesota")
print(minnesota_table)

National
col_0      count
0               
asian      15000
black      50000
hispanic   60000
other      35000
white     100000
 
Minnesota
col_0     count
0              
asian        75
black       250
hispanic    300
other       150
white       600


Chi-squared tests are based on the so-called chi-squared statistic. You calculate the chi-squared statistic with the following formula:

$$ sum(\frac{(observed-expected)^2}{expected}) $$

In the formula, observed is the actual observed count for each category and expected is the expected count based on the distribution of the population for the corresponding category. Let's calculate the chi-squared statistic for our data to illustrate:

In [ ]:
observed = minnesota_table

national_ratios = national_table/len(national)  # Get population ratios

expected = national_ratios * len(minnesota)   # Get expected counts

chi_squared_stat = (((observed-expected)**2)/expected).sum()

print(chi_squared_stat)

col_0
count    18.194805
dtype: float64


Note: The chi-squared test assumes none of the expected counts are less than 5.

Similar to the t-test where we compared the t-test statistic to a critical value based on the t-distribution to determine whether the result is significant, in the chi-square test we compare the chi-square test statistic to a critical value based on the chi-square distribution. The scipy library shorthand for the chi-square distribution is chi2. Let's use this knowledge to find the critical value for 95% confidence level and check the p-value of our result:

In [ ]:
crit = stats.chi2.ppf(q = 0.95, # Find the critical value for 95% confidence*
                      df = 4)   # Df = number of variable categories - 1

print("Critical value")
print(crit)

p_value = 1 - stats.chi2.cdf(x=chi_squared_stat,  # Find the p-value
                             df=4)
print("P value")
print(p_value)

Critical value
9.487729036781154
P value
[0.00113047]


Since our chi-squared statistic exceeds the critical value, we'd reject the null hypothesis that the two distributions are the same.

You can carry out a chi-squared goodness-of-fit test automatically using the scipy function scipy.stats.chisquare():

In [ ]:
stats.chisquare(f_obs= observed,   # Array of observed counts
                f_exp= expected)   # Array of expected counts

Power_divergenceResult(statistic=array([18.19480519]), pvalue=array([0.00113047]))

### **Chi-Squared Test of Independence**

Independence is a key concept in probability that describes a situation where knowing the value of one variable tells you nothing about the value of another. For instance, the month you were born probably doesn't tell you anything about which web browser you use, so we'd expect birth month and browser preference to be independent. On the other hand, your month of birth might be related to whether you excelled at sports in school, so month of birth and sports performance might not be independent.

The chi-squared test of independence tests whether two categorical variables are independent. The test of independence is commonly used to determine whether variables like education, political views and other preferences vary based on demographic factors like gender, race and religion. Let's generate some fake voter polling data and perform a test of independence:

In [ ]:
np.random.seed(10)

# Sample data randomly at fixed probabilities
voter_race = np.random.choice(a= ["asian","black","hispanic","other","white"],
                              p = [0.05, 0.15 ,0.25, 0.05, 0.5],
                              size=1000)

# Sample data randomly at fixed probabilities
voter_party = np.random.choice(a= ["democrat","independent","republican"],
                              p = [0.4, 0.2, 0.4],
                              size=1000)

voters = pd.DataFrame({"race":voter_race,
                       "party":voter_party})

voter_tab = pd.crosstab(voters.race, voters.party, margins = True)

voter_tab.columns = ["democrat","independent","republican","row_totals"]

voter_tab.index = ["asian","black","hispanic","other","white","col_totals"]

observed = voter_tab.iloc[0:5,0:3]   # Get table without totals for later use
voter_tab

,democrat,independent,republican,row_totals
asian,21,7,32,60
black,65,25,64,154
hispanic,107,50,94,251
other,15,8,15,38
white,189,96,212,497
col_totals,397,186,417,1000


Note that we did not use the race data to inform our generation of the party data so the variables are independent.

For a test of independence, we use the same chi-squared formula that we used for the goodness-of-fit test. The main difference is we have to calculate the expected counts of each cell in a 2-dimensional table instead of a 1-dimensional table. To get the expected count for a cell, multiply the row total for that cell by the column total for that cell and then divide by the total number of observations. We can quickly get the expected counts for all cells in the table by taking the row totals and column totals of the table, performing an outer product on them with the np.outer() function and dividing by the number of observations:

In [ ]:
expected =  np.outer(voter_tab["row_totals"][0:5],
                     voter_tab.loc["col_totals"][0:3]) / 1000

expected = pd.DataFrame(expected)

expected.columns = ["democrat","independent","republican"]
expected.index = ["asian","black","hispanic","other","white"]

expected

,democrat,independent,republican
asian,23.820,11.160,25.020
black,61.138,28.644,64.218
hispanic,99.647,46.686,104.667
other,15.086,7.068,15.846
white,197.309,92.442,207.249


Now we can follow the same steps we took before to calculate the chi-square statistic, the critical value and the p-value:



In [ ]:
chi_squared_stat = (((observed-expected)**2)/expected).sum().sum()

print(chi_squared_stat)

7.169321280162059


Note: We call .sum() twice: once to get the column sums and a second time to add the column sums together, returning the sum of the entire 2D table.



In [ ]:
crit = stats.chi2.ppf(q = 0.95, # Find the critical value for 95% confidence*
                      df = 8)   # *

print("Critical value")
print(crit)

p_value = 1 - stats.chi2.cdf(x=chi_squared_stat,  # Find the p-value
                             df=8)
print("P value")
print(p_value)

Critical value
15.50731305586545
P value
0.518479392948842


Note: The degrees of freedom for a test of independence equals the product of the number of categories in each variable minus 1. In this case we have a 5x3 table so df = 4x2 = 8.

As with the goodness-of-fit test, we can use scipy to conduct a test of independence quickly. Use stats.chi2_contingency() function to conduct a test of independence automatically given a frequency table of observed counts:

In [ ]:
stats.chi2_contingency(observed= observed)


Chi2ContingencyResult(statistic=np.float64(7.169321280162059), pvalue=np.float64(0.518479392948842), dof=8, expected_freq=array([[ 23.82 ,  11.16 ,  25.02 ],
       [ 61.138,  28.644,  64.218],
       [ 99.647,  46.686, 104.667],
       [ 15.086,   7.068,  15.846],
       [197.309,  92.442, 207.249]]))

The output shows the chi-square statistic, the p-value and the degrees of freedom followed by the expected counts.

As expected, given the high p-value, the test result does not detect a significant relationship between the variables.

### **Questions**

**1. When do we go for Chi square test ?**  

We use the Chi-squared test in two main scenarios:

**Chi-Squared Goodness-Of-Fit Test:** This test is used to determine whether the distribution of sample categorical data matches an expected distribution. For example, to check if demographic data from a sample matches a known population distribution.

**Chi-Squared Test of Independence:** This test is used to determine whether two categorical variables are independent. For example, to see if there's a relationship between a person's race and their political party affiliation.

**2. Do we use Chi square test for independence ? Explain How ?**

Yes, we absolutely use the Chi-squared test for independence! The Chi-squared Test of Independence is used to determine whether two categorical variables are independent of each other.

Here's how it works:

1. Formulate Hypotheses: The null hypothesis ($H_0$$H_0$) states that the two variables are independent (i.e., there is no relationship between them). The alternative hypothesis ($H_1$$H_1$) states that the two variables are not independent (i.e., there is a relationship).
2. Create a Contingency Table: You first organize your observed categorical data into a contingency table (also known as a cross-tabulation), which shows the frequency distribution of the variables.
3. Calculate Expected Frequencies: For each cell in the contingency table, you calculate the expected frequency under the assumption that the null hypothesis is true (i.e., the variables are independent). The expected count for a cell is calculated as (row total * column total) / grand total.
4. Calculate the Chi-Squared Statistic: The chi-squared statistic is then calculated using the formula: $$ \sum \frac{(Observed - Expected)^2}{Expected} $$$$ \sum \frac{(Observed - Expected)^2}{Expected} $$ where Observed is the actual count in each cell and Expected is the expected count for that cell.
5. Determine Degrees of Freedom: The degrees of freedom (df) for a test of independence are calculated as (number of rows - 1) * (number of columns - 1).
6. Compare to Critical Value or Find P-value: You then compare the calculated chi-squared statistic to a critical value from the chi-squared distribution (for a chosen significance level) or calculate the p-value. If the chi-squared statistic exceeds the critical value, or if the p-value is less than the significance level, you reject the null hypothesis, concluding that there is a significant relationship between the variables.



**3. Explain role of Critical value and p-value in Chi square test.**

In a Chi-square test, both the critical value and the p-value are crucial for making decisions about your hypotheses:

* Critical Value: The critical value is a threshold
determined by the chosen significance level (alpha, often 0.05) and the degrees of freedom of your test. It marks the boundary beyond which we consider a result statistically significant. If your calculated Chi-square statistic is greater than the critical value, it suggests that the observed differences are unlikely to have occurred by random chance alone, leading you to reject the null hypothesis.

* P-value: The p-value is the probability of observing a Chi-square statistic as extreme as, or more extreme than, the one calculated from your sample data, assuming the null hypothesis is true. A small p-value (typically less than your chosen significance level, e.g., p < 0.05) indicates that the observed data is very unlikely if the null hypothesis were true. This leads to the rejection of the null hypothesis. Conversely, a large p-value suggests that the observed data is consistent with the null hypothesis, and you would fail to reject it.

**4. Can we use chi square test to show existence of correlation between 2 features in a dataset ?  How will you interpret test result to indicate correlation presence.**

Yes, the Chi-squared test can be used to show the existence of a relationship or association between two categorical features in a dataset. While it doesn't measure the strength or direction of a linear 'correlation' like Pearson's r (which is for numerical variables), it determines if the observed frequencies of categories for two variables are significantly different from what would be expected if the variables were independent.

Here's how you interpret the test result to indicate the presence of an association:

* Null Hypothesis ($H_0$$H_0$): The two categorical variables are independent (i.e., there is no relationship or association between them).
* Alternative Hypothesis ($H_1$$H_1$): The two categorical variables are not independent (i.e., there is a relationship or association between them).

Interpretation based on p-value:
* If the p-value is less than your chosen significance level (e.g., $\alpha = 0.05$$\alpha = 0.05$): You reject the null hypothesis. This indicates that there is a statistically significant relationship or association between the two categorical variables. In simpler terms, the distribution of one variable is dependent on the other.
* If the p-value is greater than your chosen significance level: You fail to reject the null hypothesis. This suggests that there is not enough evidence to conclude a statistically significant relationship or association between the two variables. They are likely independent.

Interpretation based on critical value (less common for direct interpretation but still relevant):
* If the calculated Chi-squared statistic is greater than the critical value: You reject the null hypothesis, implying an association between the variables.
* If the calculated Chi-squared statistic is less than the critical value: You fail to reject the null hypothesis, suggesting independence.

**5. What is degrees of freedom and how to calculate it while performing chi square test.**

Degrees of freedom (df) in statistics refers to the number of independent pieces of information that go into the calculation of a statistic. In simpler terms, it's the number of values in the final calculation of a statistic that are free to vary.

For Chi-square tests, the calculation of degrees of freedom depends on the specific type of test:

* For the Chi-Squared Goodness-Of-Fit Test: The degrees of freedom are calculated as the number of categories - 1.
Example from the notebook: For the goodness-of-fit test comparing Minnesota demographics to national demographics, there are 5 categories (asian, black, hispanic, other, white). So, df = 5 - 1 = 4.

* For the Chi-Squared Test of Independence: The degrees of freedom are calculated as (number of rows - 1) * (number of columns - 1).
Example from the notebook: For the test of independence between voter race and party affiliation, we had a 5x3 table (5 races, 3 parties). So, df = (5 - 1) * (3 - 1) = 4 * 2 = 8.
